In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

def load_saved_matrix(modality, model_name, state):
    """
    Charge un fichier .npy en utilisant le pattern exact de votre dossier TFE_Data.
    Exemple de 'state': 'pca_64', 'grp_128', 'raw'
    """
    filename = f"X_{modality}_{model_name}_{state}_Flickr8k.npy"
    path = os.path.join(DATA_DIR, filename)
    
    if os.path.exists(path):
        return np.load(path)
    else:
        # Petit hack au cas où le nom varie légèrement
        raise FileNotFoundError(f"Fichier manquant : {filename} dans {DATA_DIR}")

def load_raw_matrix(modality, model_name):
    return load_saved_matrix(modality, model_name, "raw")

In [2]:
def run_exhaustive_xai_summary(modality, model_list, methods=['pca', 'grp'], dims=[16, 32, 64, 128, 256, 512], ks=[5, 10, 50]):
    report_data = []
    
    for model_name in model_list:
        print(f"\n--- Analyse de {modality.upper()} : {model_name} ---")
        
        try:
            # 1. Charger le RAW (La référence de vérité)
            raw_matrix = load_raw_matrix(modality, model_name)
            
            # On prend un échantillon de 100 pour la rapidité
            n_test = min(100, len(raw_matrix))
            raw_sample = raw_matrix[:n_test]
            
            # Calculer TOUTES les similarités RAW d'un coup (Matrice vs Matrice)
            sim_raw = cosine_similarity(raw_sample, raw_matrix)
        except Exception as e:
            print(f"⚠️ Erreur chargement RAW pour {model_name}: {e}")
            continue
            
        for method in methods:
            for dim in dims:
                state = f"{method}_{dim}"
                try:
                    # 2. Charger le REDUIT
                    red_matrix = load_saved_matrix(modality, model_name, state)
                    red_sample = red_matrix[:n_test]
                    
                    # Calculer TOUTES les similarités REDUITES
                    sim_red = cosine_similarity(red_sample, red_matrix)
                    
                    # 3. Calculer la préservation pour chaque K
                    stats_k = {k: [] for k in ks}
                    
                    for i in range(n_test):
                        for k in ks:
                            # On récupère les indices des Top K (on exclut l'élément lui-même)
                            idx_raw = np.argsort(sim_raw[i])[-k-1:-1]
                            idx_red = np.argsort(sim_red[i])[-k-1:-1]
                            
                            # Intersection
                            overlap = len(set(idx_raw).intersection(set(idx_red)))
                            stats_k[k].append(overlap / k)
                    
                    # Ajouter au rapport
                    report_data.append({
                        "Model": model_name,
                        "Method": method.upper(),
                        "Dim": dim,
                        "P@5": np.mean(stats_k[5]),
                        "P@10": np.mean(stats_k[10]),
                        "P@50": np.mean(stats_k[50])
                    })
                    print(f"  ✅ {state} traité.")
                    
                except:
                    continue # Ignore si la combinaison n'existe pas sur le disque
                
    return pd.DataFrame(report_data)

In [3]:
# Listes basées sur vos fichiers réels
v_models = ["resnet50", "vit", "deit", "mobilenet_v3", "clip_vision"]
t_models = ["bert", "roberta", "gpt2", "rnn", "clip_text"]

print("Génération du rapport Vision...")
vision_results = run_exhaustive_xai_summary("vision", v_models)

print("\nGénération du rapport Texte...")
text_results = run_exhaustive_xai_summary("text", t_models)

# Affichage des meilleurs résultats (triés par stabilité P@5)
print("\nTOP 10 STABILITÉ VISION")
display(vision_results.sort_values("P@5", ascending=False).head(10))

print("\nTOP 10 STABILITÉ TEXTE")
display(text_results.sort_values("P@5", ascending=False).head(10))

Génération du rapport Vision...

--- Analyse de VISION : resnet50 ---
⚠️ Erreur chargement RAW pour resnet50: name 'DATA_DIR' is not defined

--- Analyse de VISION : vit ---
⚠️ Erreur chargement RAW pour vit: name 'DATA_DIR' is not defined

--- Analyse de VISION : deit ---
⚠️ Erreur chargement RAW pour deit: name 'DATA_DIR' is not defined

--- Analyse de VISION : mobilenet_v3 ---
⚠️ Erreur chargement RAW pour mobilenet_v3: name 'DATA_DIR' is not defined

--- Analyse de VISION : clip_vision ---
⚠️ Erreur chargement RAW pour clip_vision: name 'DATA_DIR' is not defined

Génération du rapport Texte...

--- Analyse de TEXT : bert ---
⚠️ Erreur chargement RAW pour bert: name 'DATA_DIR' is not defined

--- Analyse de TEXT : roberta ---
⚠️ Erreur chargement RAW pour roberta: name 'DATA_DIR' is not defined

--- Analyse de TEXT : gpt2 ---
⚠️ Erreur chargement RAW pour gpt2: name 'DATA_DIR' is not defined

--- Analyse de TEXT : rnn ---
⚠️ Erreur chargement RAW pour rnn: name 'DATA_DIR' is not def

KeyError: 'P@5'